# 17. 확장된 라이브러리(10개 규칙)로 전체 재검증

## 이번 노트북에서 할 것
- Held-out 커버리지 재측정 (기존 227개 → 얼마나 늘었는지, Michael_acceptor_1/
  acid_halide/catechol/Thiocarbonyl_group 포함)
- 단일 문제 분자 표본 재검증 (기존 58%+36%=94% 개선이 어떻게 바뀌는지)
- 3-endpoint(Tox21/Ames/hERG) 통계 재계산 (167개 규모로 재시도)
- 다중 문제 분자 재검증 (새 규칙 포함 시 더 많은 조합 가능)

## 간략한 정리 (16까지)
- 중대 버그 수정: 이름 불일치로 죽어있던 규칙 발견(michael_acceptor, thiourea,
  acyl_halide, phenol, amide) → thiourea/phenol/amide는 FilterCatalog 대응
  규칙 없어 완전 제거(과도한 일반화였음 인정), michael_acceptor/acyl_halide는
  이름만 수정(Michael_acceptor_1, acid_halide)해서 복구
- 신규 원자편집(atom_edit) 방식 도입: catechol(문맥의존 재귀SMARTS),
  Thiocarbonyl_group(고리내부 원자) - RWMol 기반, propose_fix가 edit_method로 분기
- 현재 라이브러리 10개 규칙, 전수 검증 완료(전체 7823개 분자 기준 전부 0회 아님)

## 다음에 해야 할 것 (오늘 끝나면)
- 18번: quinone_A 재도전(atom_edit 방식으로), thiol_1/thiol_2 관계 확인
- 전체 held-out(1174개) 최종 대량 실행
- README 정리

In [1]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 12.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 191 (delta 92), reused 131 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 581.66 KiB | 25.29 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hykyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib
import numpy as np
import pandas as pd
from scipy import stats
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, QED
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
from src.tools.atom_editor import apply_atom_edit_from_rule

data = load_tox21_clean()

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

print("도구 로드 확인 완료")

[04:02:31] WARNING: not removing hydrogen atom without neighbors
[04:02:32] Explicit valence for atom # 8 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 3 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 9 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 5 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 16 Al, 6, is greater than permitted
[04:02:32] Explicit valence for atom # 20 Al, 6, is greater than permitted
[04:02:32] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개
도구 로드 확인 완료


In [5]:
# 셀 5 — Tox21/Ames/hERG baseline 재학습
X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']
classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf
print("Tox21 baseline 완료")

from tdc.single_pred import Tox

def prepare_split_generic(df):
    df = df.copy()
    df['mol_valid'] = df['Drug'].apply(lambda s: Chem.MolFromSmiles(s) is not None)
    df_clean = df[df['mol_valid']].reset_index(drop=True)
    X = np.stack(df_clean['Drug'].apply(smiles_to_ecfp).values)
    y = df_clean['Y'].values
    return X, y, df_clean['Drug'].values

ames_split = Tox(name='AMES').get_split()
X_train_ames, y_train_ames, _ = prepare_split_generic(ames_split['train'])
ames_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
ames_clf.fit(X_train_ames, y_train_ames)
print("Ames baseline 완료")

herg_split = Tox(name='hERG').get_split()
X_train_herg, y_train_herg, _ = prepare_split_generic(herg_split['train'])
herg_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
herg_clf.fit(X_train_herg, y_train_herg)
print("hERG baseline 완료")

def predict_ames(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return ames_clf.predict_proba(smiles_to_ecfp(smiles).reshape(1, -1))[0][1]

def predict_herg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return herg_clf.predict_proba(smiles_to_ecfp(smiles).reshape(1, -1))[0][1]

def predict_tox21_avg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    return np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols])

Tox21 baseline 완료


Downloading...
100%|██████████| 344k/344k [00:00<00:00, 523kiB/s]
Loading...
Done!
Downloading...


Ames baseline 완료


100%|██████████| 50.2k/50.2k [00:00<00:00, 231kiB/s] 
Loading...
Done!
[04:03:37] WARNING: not removing hydrogen atom without neighbors
[04:03:37] WARNING: not removing hydrogen atom without neighbors
[04:03:37] WARNING: not removing hydrogen atom without neighbors
[04:03:37] WARNING: not removing hydrogen atom without neighbors


hERG baseline 완료


In [6]:
# 셀 6 — Qwen 연결
from openai import OpenAI
dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [7]:
count_known_v4 = 0
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 1:
        count_known_v4 += 1

print(f"이전(9개 규칙, 이름 오류 포함): 227개")
print(f"실제 유효했던 규칙 기준(6개): ??")  # 참고용, 아래서 재계산
print(f"현재(10개 규칙, 전부 검증됨): {count_known_v4}개 / {len(data['smiles_test'])}개 ({count_known_v4/len(data['smiles_test'])*100:.1f}%)")

이전(9개 규칙, 이름 오류 포함): 227개
실제 유효했던 규칙 기준(6개): ??
현재(10개 규칙, 전부 검증됨): 293개 / 1174개 (25.0%)


In [8]:
import random
random.seed(42)

single_known_v2 = []
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count == 1:
        single_known_v2.append(s)

sample_single_v2 = random.sample(single_known_v2, min(50, len(single_known_v2)))
print(f"단일 문제 분자 풀: {len(single_known_v2)}개, 표본: {len(sample_single_v2)}개")

rule_based_results_v2 = []
for smi in sample_single_v2:
    result = iterative_fix_loop(smi, max_iterations=10)
    rule_based_results_v2.append({"smiles": smi, "status": result['status'], "steps": len(result['history'])-1})

from collections import Counter
status_counts_v2 = Counter(r['status'] for r in rule_based_results_v2)
print("\n규칙기반 결과 (50개 표본):")
for status, count in status_counts_v2.items():
    print(f"  {status}: {count}개 ({count/len(rule_based_results_v2)*100:.1f}%)")

단일 문제 분자 풀: 260개, 표본: 50개


[04:05:03] Incomplete atom labelling, cannot make bond



규칙기반 결과 (50개 표본):
  success: 26개 (52.0%)
  no_known_fix: 10개 (20.0%)
  stuck: 14개 (28.0%)


In [9]:
total_v2 = len(rule_based_results_v2)
success_v2 = sum(1 for r in rule_based_results_v2 if r['status'] == 'success')
partial_v2 = sum(1 for r in rule_based_results_v2 if r['status'] == 'no_known_fix' and r['steps'] >= 1)

print(f"완전 해결(success): {success_v2}개 ({success_v2/total_v2*100:.1f}%)")
print(f"부분 진전(1단계 이상): {partial_v2}개 ({partial_v2/total_v2*100:.1f}%)")
print(f"최소 1단계 이상 개선된 비율: {(success_v2+partial_v2)/total_v2*100:.1f}%")

완전 해결(success): 26개 (52.0%)
부분 진전(1단계 이상): 10개 (20.0%)
최소 1단계 이상 개선된 비율: 72.0%


In [10]:
stuck_cases = [r for r in rule_based_results_v2 if r['status'] == 'stuck']
print(f"stuck 케이스: {len(stuck_cases)}개\n")

for r in stuck_cases[:5]:
    detail = iterative_fix_loop(r['smiles'], max_iterations=10)
    print(f"분자: {r['smiles'][:50]}")
    for h in detail['history']:
        print(f"  {h}")
    print()

stuck 케이스: 14개

분자: CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C
  {'step': 0, 'smiles': 'CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [15, 16]}]}

분자: CC(CC(C)C)=NO
  {'step': 0, 'smiles': 'CC(CC(C)C)=NO', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [1, 6]}, {'rule_name': 'oxime_1', 'atom_indices': [1, 6, 7]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [6, 7]}]}

분자: CC1(C)S[C@@H]2[C@H](/N=C/N3CCCCCC3)C(=O)N2[C@H]1C(
  {'step': 0, 'smiles': 'CC1(C)S[C@@H]2[C@H](/N=C/N3CCCCCC3)C(=O)N2[C@H]1C(=O)O', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [6, 7]}, {'rule_name': 'imine_2', 'atom_indices': [6, 7, 8]}]}

분자: CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(=N)N)NC(=O)[C
  {'step': 0, 'smiles': 'CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(=N)N)NC(=O)[C@H](CC(C)C)NC(=O)CNC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)[C@H](CO)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@@H]1CCC(=O)N1', 'problems': [{'rule_name': 'Aliphatic_lon

In [11]:
info_imine1 = get_replacement_candidates("imine_1")
print(info_imine1['problem_smarts'])

test_general_imine = "CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C"
mol_test = Chem.MolFromSmiles(test_general_imine)
pattern = Chem.MolFromSmarts(info_imine1['problem_smarts'])
print("일반 이민(OH 없음) 매치 여부:", mol_test.HasSubstructMatch(pattern))

C=N[OX2H1]
일반 이민(OH 없음) 매치 여부: False


In [12]:
count_oxime = 0
count_general_imine = 0

oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")

for s in data['smiles_test']:
    p = detect_toxicophores(s)
    has_imine1 = any(x['rule_name'] == 'imine_1' for x in p)
    if has_imine1:
        mol = Chem.MolFromSmiles(s)
        if mol.HasSubstructMatch(oxime_pattern):
            count_oxime += 1
        else:
            count_general_imine += 1

print(f"imine_1 중 옥심(OH 있음): {count_oxime}개")
print(f"imine_1 중 일반 이민(OH 없음): {count_general_imine}개")

imine_1 중 옥심(OH 있음): 3개
imine_1 중 일반 이민(OH 없음): 34개


In [13]:
!cat src/tools/toxicophore_detector.py

from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()

def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []
    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            results.append({
                "rule_name": entry.GetDescription(),
                "atom_indices": atom_indices,
            })
    return results


In [14]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH)과 일반 이민(C=N-R)을 모두 포함하는 넓은 카테고리이므로,
    실제 매치된 부분이 옥심 패턴을 포함하는지 확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    return "imine_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/일반이민 하위형으로 세분화하여 반환한다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []
    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })
    return results

Overwriting src/tools/toxicophore_detector.py


In [15]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
    "imine_1_general": {
        "problem_smarts": "C=N",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [16]:
import importlib
import src.tools.toxicophore_detector
import src.tools.replacement_library
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.molecule_editor import propose_fix

# 아까 stuck났던 케이스들 재확인
print(detect_toxicophores("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C"))
print(detect_toxicophores("CC(CC(C)C)=NO"))

print(propose_fix("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C", "imine_1_general", candidate_idx=0))
print(propose_fix("CC(CC(C)C)=NO", "imine_1_oxime", candidate_idx=0))

[{'rule_name': 'imine_1_general', 'atom_indices': [15, 16]}]
[{'rule_name': 'imine_1_oxime', 'atom_indices': [1, 6]}, {'rule_name': 'oxime_1', 'atom_indices': [1, 6, 7]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [6, 7]}]
None
None


In [17]:
core_check_general = find_core_and_target("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C", "imine_1_general")
print("general core:", core_check_general)

core_check_oxime = find_core_and_target("CC(CC(C)C)=NO", "imine_1_oxime")
print("oxime core:", core_check_oxime)

general core: None
oxime core: None


In [20]:
from rdkit.Chem import rdMMPA

mol_general = Chem.MolFromSmiles("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C")
frags_general = rdMMPA.FragmentMol(mol_general, maxCuts=1, resultsAsMols=False)
print("=== general (maxCuts=1) ===")
for core, chain in frags_general:
    print(f"core: {core}, chain: {chain}")

mol_oxime = Chem.MolFromSmiles("CC(CC(C)C)=NO")
frags_oxime = rdMMPA.FragmentMol(mol_oxime, maxCuts=1, resultsAsMols=False)
print("\n=== oxime (maxCuts=1) ===")
for core, chain in frags_oxime:
    print(f"core: {core}, chain: {chain}")

=== general (maxCuts=1) ===
core: , chain: CC(N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C)[*:1].C[*:1]
core: , chain: CC(C)(C)N=C1SCN(c2ccccc2)C(=O)N1[*:1].CC(C)[*:1]
core: , chain: CC(C)N1C(=O)N([*:1])CSC1=NC(C)(C)C.c1ccc([*:1])cc1
core: , chain: CC(C)(C)[*:1].CC(C)N1C(=O)N(c2ccccc2)CSC1=N[*:1]
core: , chain: CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)[*:1].C[*:1]

=== oxime (maxCuts=1) ===
core: , chain: CC(C)CC(=NO)[*:1].C[*:1]
core: , chain: CC(=NO)[*:1].CC(C)C[*:1]
core: , chain: CC(C)[*:1].CC(C[*:1])=NO
core: , chain: CC(CC(C)[*:1])=NO.C[*:1]


In [21]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[info["target_idx_in_pattern"]]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[info["target_idx_in_pattern"]]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        # target_idx_pair_in_pattern: (원자1의 패턴내 위치, 원자2의 패턴내 위치)
        idx1 = match[info["target_idx_pair_in_pattern"][0]]
        idx2 = match[info["target_idx_pair_in_pattern"][1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)  # sanitize가 알아서 필요한 H를 다시 계산하도록
            atom.SetNoImplicit(True)

        # oxime 후보처럼, 결합 환원 후 특정 원자를 추가로 치환해야 하는 경우
        if candidate.get("also_replace_idx_in_pattern") is not None:
            also_idx = match[candidate["also_replace_idx_in_pattern"]]
            also_atom = rwmol.GetAtomWithIdx(also_idx)
            if candidate.get("also_remove_atom"):
                # O를 제거하는 경우 (니트릴 후보처럼 구조가 크게 바뀌는 경우는 별도 처리 필요)
                pass  # 이번 케이스에서는 사용 안 함
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)
    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [22]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [23]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

print("general:", propose_fix("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C", "imine_1_general", candidate_idx=0))
print("oxime:", propose_fix("CC(CC(C)C)=NO", "imine_1_oxime", candidate_idx=0))

general: {'new_smiles': 'CC(C)N1[C]([N]C(C)(C)C)SCN(c2ccccc2)C1=O', 'candidate_used': 'amine (reduced)', 'rationale': '일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요', 'is_valid': True}
oxime: {'new_smiles': 'C[C](CC(C)C)[N]O', 'candidate_used': 'amine (reduced)', 'rationale': '옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 경로를 제거함', 'is_valid': True}


In [24]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[info["target_idx_in_pattern"]]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[info["target_idx_in_pattern"]]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        idx1 = match[info["target_idx_pair_in_pattern"][0]]
        idx2 = match[info["target_idx_pair_in_pattern"][1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    # 유효성 검증 강화: 파싱 가능 여부뿐 아니라, [C]/[N]처럼 암묵적 수소가
    # 비정상적으로 억제된 원자가 남아있는지도 확인
    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        for atom in check_mol.GetAtoms():
            if atom.GetNoImplicit() and atom.GetSymbol() in ('C', 'N', 'O') and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4:
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [25]:
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

result_general = propose_fix("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C", "imine_1_general", candidate_idx=0)
print("general:", result_general)

result_oxime = propose_fix("CC(CC(C)C)=NO", "imine_1_oxime", candidate_idx=0)
print("oxime:", result_oxime)

general: {'new_smiles': 'CC(C)N1C(=O)N(c2ccccc2)CSC1NC(C)(C)C', 'candidate_used': 'amine (reduced)', 'rationale': '일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요', 'is_valid': True}
oxime: {'new_smiles': 'CC(C)CC(C)NO', 'candidate_used': 'amine (reduced)', 'rationale': '옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 경로를 제거함', 'is_valid': True}


In [26]:
!git add src/tools/atom_editor.py src/tools/replacement_library.py src/tools/toxicophore_detector.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py
	modified:   src/tools/toxicophore_detector.py



In [27]:
!git commit -m "Fix critical bug in reduce_bond edit: SetNoImplicit(True) was blocking automatic H recalculation, producing invalid atoms like [C]/[N]; also strengthen is_valid check to catch this class of error. Add imine_1 subclassification (oxime vs general) via toxicophore_detector post-processing, both now use atom_edit with new reduce_bond type. Library now has 11 rules, all verified."
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main fd12991] Fix critical bug in reduce_bond edit: SetNoImplicit(True) was blocking automatic H recalculation, producing invalid atoms like [C]/[N]; also strengthen is_valid check to catch this class of error. Add imine_1 subclassification (oxime vs general) via toxicophore_detector post-processing, both now use atom_edit with new reduce_bond type. Library now has 11 rules, all verified.
 3 files changed, 63 insertions(+), 12 deletions(-)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 2.14 KiB | 2.14 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/Dec32th/laidd-2026.git
   fba049e..fd12991  main -> main


In [28]:
all_rule_names_v5 = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
print("현재 라이브러리:", all_rule_names_v5, f"({len(all_rule_names_v5)}개)")

rule_hit_counts_v5 = {name: 0 for name in all_rule_names_v5}
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    found_names = set(x['rule_name'] for x in p)
    for rule in all_rule_names_v5:
        if rule in found_names:
            rule_hit_counts_v5[rule] += 1

print("\n=== 규칙별 발동 횟수 (test set 1174개 기준) ===")
for rule, count in rule_hit_counts_v5.items():
    print(f"{rule}: {count}회")

count_known_v5 = sum(1 for s in data['smiles_test'] if any(get_replacement_candidates(x['rule_name']) is not None for x in detect_toxicophores(s)))
print(f"\n전체 커버리지: {count_known_v5}개 / {len(data['smiles_test'])}개 ({count_known_v5/len(data['smiles_test'])*100:.1f}%)")

현재 라이브러리: ['nitro_group', 'aldehyde', 'Michael_acceptor_1', 'acid_halide', 'alkyl_halide', 'aniline', 'Sulfonic_acid_2', 'imine_1_oxime', 'imine_1_general', 'catechol', 'Thiocarbonyl_group'] (11개)

=== 규칙별 발동 횟수 (test set 1174개 기준) ===
nitro_group: 49회
aldehyde: 27회
Michael_acceptor_1: 50회
acid_halide: 3회
alkyl_halide: 53회
aniline: 44회
Sulfonic_acid_2: 34회
imine_1_oxime: 3회
imine_1_general: 34회
catechol: 16회
Thiocarbonyl_group: 13회

전체 커버리지: 293개 / 1174개 (25.0%)
